# 16x_SHAP_candidate_interpretation_payment_removed_retry_260516

Retry notebook for payment-removed SHAP. The payment-removal input gate runs before any model fit, SHAP calculation, or figure generation.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib, json, os, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.ensemble import HistGradientBoostingClassifier
import shap
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
STEP = '16x_SHAP_candidate_interpretation_260516'
RETRY_STEP = '16x_SHAP_candidate_interpretation_payment_removed_retry_260516'
PAYMENT_FEATURES = ['payment_is_mobile', 'payment_is_pc', 'payment_is_android', 'payment_is_ios']
EXPECTED_COUNTS = {'overall_with_promotion': 76, 'overall_without_promotion': 75, 'promotion_only': 75, 'nonpromotion_only': 75}
SCOPES = list(EXPECTED_COUNTS.keys())

START = Path.cwd().resolve()
ROOT = None
PARK = None
for cand in [START] + list(START.parents):
    if cand.name == 'park.ingyeom' and (cand / 'note.md').exists():
        PARK = cand.resolve(); ROOT = cand.parent.resolve(); break
    if (cand / 'park.ingyeom' / 'note.md').exists():
        ROOT = cand.resolve(); PARK = (cand / 'park.ingyeom').resolve(); break
assert PARK is not None and PARK.name == 'park.ingyeom', f'Could not locate park.ingyeom from {START}'

NOTE = PARK / 'note.md'
NB_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT = PARK / 'reports' / 'interpretation' / STEP
FIG = PARK / 'reports' / 'figures' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
for p in [NB_PATH.parent, OUT, FIG, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def inside_park(path):
    return str(Path(path).resolve()).lower().startswith(str(PARK).lower())

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_csv(path):
    return pd.read_csv(path)

def write_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def final_checks_pass(path):
    df = pd.read_csv(path)
    status_cols = [c for c in df.columns if c.lower() in {'status', 'result', 'check_status'}]
    if status_cols:
        vals = df[status_cols[0]].astype(str).str.lower()
        return not vals.str.fullmatch('fail|failed|error|critical_fail').any()
    return True

def scope_df(df, scope):
    if scope == 'promotion_only':
        return df[df['is_promotion'] == 1].copy()
    if scope == 'nonpromotion_only':
        return df[df['is_promotion'] == 0].copy()
    return df.copy()

def family(feature):
    if feature.startswith('payment_is_'): return 'payment_device_proxy_removed'
    if feature in {'is_user_verified'}: return 'auth_proxy_caveat'
    if feature in {'age_group','is_female','is_male'}: return 'demographic_proxy_caveat'
    if feature in {'is_standard','is_premium','is_basic'}: return 'membership_context'
    if feature == 'is_promotion': return 'acquisition_split_key'
    if feature.startswith('reg_'): return 'registration_context'
    if any(x in feature for x in ['watch','session','retention','diff_between','only_w','cold_start','recency','gap','inactive']): return 'usage_retention_behavior'
    if any(x in feature for x in ['drama','comedy','romance','horror','documentary','action','family','thriller','sf','historical','other','movie','release','genre']): return 'content_preference_context'
    return 'other_feature_family'

def make_model(model_name, params):
    if model_name == 'LightGBM':
        from lightgbm import LGBMClassifier
        p = dict(params)
        p.update({'random_state': RANDOM_STATE, 'n_jobs': -1, 'objective': 'binary', 'verbosity': -1})
        return LGBMClassifier(**p)
    if model_name == 'CatBoost':
        from catboost import CatBoostClassifier
        p = dict(params) if params else {'iterations': 300, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
        p.update({'random_seed': RANDOM_STATE, 'verbose': False, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'allow_writing_files': False})
        return CatBoostClassifier(**p)
    p = dict(params) if params else {'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}
    p.update({'random_state': RANDOM_STATE})
    return HistGradientBoostingClassifier(**p)

def shap_values_2d(model, X):
    explainer = shap.TreeExplainer(model)
    vals = explainer.shap_values(X)
    if isinstance(vals, list): vals = vals[1] if len(vals) > 1 else vals[0]
    vals = np.asarray(vals)
    if vals.ndim == 3: vals = vals[:, :, 1] if vals.shape[2] > 1 else vals[:, :, 0]
    base = getattr(explainer, 'expected_value', np.nan)
    if isinstance(base, (list, np.ndarray)): base = base[1] if len(base) > 1 else base[0]
    return vals, base

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()

required = {
    '06x_expanded_dataset.csv': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_expanded_dataset.csv',
    '06x_model_feature_lists.csv': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_model_feature_lists.csv',
    '06x_final_checks.csv': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515' / '06x_final_checks.csv',
    '12x_candidate_selection_by_scope.csv': PARK / 'reports' / 'models' / '12x_model_family_comparison_260516' / '12x_candidate_selection_by_scope.csv',
    '12x_final_checks.csv': PARK / 'reports' / 'models' / '12x_model_family_comparison_260516' / '12x_final_checks.csv',
    '14x_candidate_recommendation_summary.csv': PARK / 'reports' / 'models' / '14x_lightweight_candidate_tuning_260516' / '14x_candidate_recommendation_summary.csv',
    '14x_best_params_by_scope.csv': PARK / 'reports' / 'models' / '14x_lightweight_candidate_tuning_260516' / '14x_best_params_by_scope.csv',
    '14x_final_checks.csv': PARK / 'reports' / 'models' / '14x_lightweight_candidate_tuning_260516' / '14x_final_checks.csv',
    '15x_expanded_no_payment_device_feature_list.csv': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_expanded_no_payment_device_feature_list.csv',
    '15x_SHAP_handoff_for_16x.csv': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_SHAP_handoff_for_16x.csv',
    '15x_recommendation_for_canonical_feature_contract.csv': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_recommendation_for_canonical_feature_contract.csv',
    '15x_final_checks.csv': PARK / 'reports' / 'audits' / '15x_payment_device_sensitivity_260516' / '15x_final_checks.csv',
}
preflight_rows = []
for name, path in required.items():
    exists = path.exists()
    fc = final_checks_pass(path) if exists and name.endswith('final_checks.csv') else None
    preflight_rows.append({'file_name': name, 'path': str(path), 'exists': exists, 'final_checks_pass': fc, 'status': 'PASS' if exists and fc is not False else 'FAIL'})
preflight = pd.DataFrame(preflight_rows)
write_csv(preflight, '16x_preflight_input_validation.csv')
input_stop = not preflight['status'].eq('PASS').all()

manifest_path = OUT / '16x_deleted_failed_payment_not_removed_manifest.csv'
if not manifest_path.exists():
    pd.DataFrame(columns=['target_path','existed_before_delete','action','status','reason']).to_csv(manifest_path, index=False, encoding='utf-8-sig')

source_files = [p for p in required.values() if p.exists()]
fp_before = pd.DataFrame([{'source_file': str(p), 'sha256_before': sha256_file(p), 'size_before': p.stat().st_size} for p in source_files])

final_status = 'PASS'
stop_reason = ''
if input_stop:
    final_status = 'FAIL'
    stop_reason = 'required input file missing or upstream final_checks failed'

dataset = read_csv(required['06x_expanded_dataset.csv']) if not input_stop else pd.DataFrame()
feature_lists = read_csv(required['06x_model_feature_lists.csv']) if not input_stop else pd.DataFrame()
feature_policy = read_csv(required['15x_expanded_no_payment_device_feature_list.csv']) if not input_stop else pd.DataFrame()
handoff15 = read_csv(required['15x_SHAP_handoff_for_16x.csv']) if not input_stop else pd.DataFrame()
rec14 = read_csv(required['14x_candidate_recommendation_summary.csv']) if not input_stop else pd.DataFrame()
best14 = read_csv(required['14x_best_params_by_scope.csv']) if not input_stop else pd.DataFrame()
candidate12 = read_csv(required['12x_candidate_selection_by_scope.csv']) if not input_stop else pd.DataFrame()

gate_rows = []
features_by_scope = {}
if not input_stop:
    original_expanded = feature_lists[(feature_lists['feature_set_name'] == 'expanded_feature_set') & (feature_lists['use_as_feature'].astype(str).str.lower() == 'yes')]['safe_model_feature_name'].astype(str).tolist()
    original_count = len(original_expanded)
    for scope in SCOPES:
        rows = feature_policy[(feature_policy['dataset_scope'] == scope) & (feature_policy['included_as_feature'].astype(str).str.lower() == 'yes')]
        features = rows['feature_name'].astype(str).tolist()
        features_by_scope[scope] = features
        present = {pf: pf in features for pf in PAYMENT_FEATURES}
        count_ok = len(features) == EXPECTED_COUNTS[scope]
        absent_ok = not any(present.values())
        missing_cols = [f for f in features if f not in dataset.columns]
        status = 'PASS' if count_ok and absent_ok and not missing_cols else 'FAIL'
        gate_rows.append({'dataset_scope': scope, 'original_expanded_feature_count': original_count, 'expected_payment_removed_feature_count': EXPECTED_COUNTS[scope], 'actual_SHAP_input_feature_count': len(features), 'payment_is_mobile_present': present['payment_is_mobile'], 'payment_is_pc_present': present['payment_is_pc'], 'payment_is_android_present': present['payment_is_android'], 'payment_is_ios_present': present['payment_is_ios'], 'missing_dataset_columns': ';'.join(missing_cols), 'status': status})
gate = pd.DataFrame(gate_rows)
write_csv(gate, '16x_payment_removed_input_gate.csv')
gate_pass = (not input_stop) and len(gate) == 4 and gate['status'].eq('PASS').all()
if not gate_pass and final_status != 'FAIL':
    final_status = 'FAIL'
    stop_reason = 'payment_removed_input_gate failed before model fit and SHAP'

payment_audit_rows = []
candidate_rows = []
models_to_run = []
if not input_stop:
    original_set = set(original_expanded)
    for scope, features in features_by_scope.items():
        for pf in PAYMENT_FEATURES:
            payment_audit_rows.append({'dataset_scope': scope, 'removed_feature': pf, 'existed_in_original_expanded': pf in original_set, 'removed_from_SHAP_input': pf not in features, 'status': 'PASS' if pf in original_set and pf not in features else 'FAIL'})
        extra_removed = sorted((original_set - set(features)) - set(PAYMENT_FEATURES))
        extra_removed = [f for f in extra_removed if not (f == 'is_promotion' and scope != 'overall_with_promotion')]
        model_name = 'LightGBM'
        if scope == 'nonpromotion_only':
            model_name = 'CatBoost'
        params = {}
        params_source = '12x_fixed_parameter_fallback_or_library_default'
        b14 = best14[(best14['feature_set_name'] == 'expanded_feature_set') & (best14['dataset_scope'] == scope) & (best14['model_name'] == model_name)]
        if len(b14) > 0:
            params = json.loads(str(b14.iloc[0]['best_params_json']))
            params_source = '14x_tuned_params_reused_without_new_tuning'
        row_count = len(scope_df(dataset, scope))
        will_run = 'yes' if gate_pass else 'no'
        reason = 'payment gate PASS; fitted SHAP explanation only' if gate_pass else 'payment gate failed; SHAP blocked before model fit'
        candidate_rows.append({'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': model_name, 'source_step': '15x no-payment feature list plus 14x candidate params when available', 'selection_basis': 'retry requires payment-removed expanded SHAP; no new tuning', 'row_count': row_count, 'feature_count': len(features), 'payment_features_removed yes/no': 'yes' if all(pf not in features for pf in PAYMENT_FEATURES) else 'no', 'will_run_SHAP yes/no': will_run, 'reason': reason, 'params_source': params_source, 'non_payment_extra_removed': ';'.join(extra_removed)})
        models_to_run.append({'scope': scope, 'features': features, 'model_name': model_name, 'params': params, 'params_source': params_source, 'extra_removed': extra_removed})
payment_audit = pd.DataFrame(payment_audit_rows)
candidate_plan = pd.DataFrame(candidate_rows)
write_csv(payment_audit, '16x_payment_removed_feature_audit.csv')
write_csv(candidate_plan, '16x_SHAP_candidate_plan.csv')

fonts = [f.name for f in font_manager.fontManager.ttflist]
font_name = 'Malgun Gothic' if 'Malgun Gothic' in fonts else ('Noto Sans CJK KR' if 'Noto Sans CJK KR' in fonts else 'DejaVu Sans')
rcParams['font.family'] = font_name
rcParams['axes.unicode_minus'] = False
write_csv(pd.DataFrame([{'os_name': os.name, 'selected_font': font_name, 'malgun_gothic_available': 'Malgun Gothic' in fonts, 'axes_unicode_minus': rcParams['axes.unicode_minus'], 'status': 'PASS'}]), '16x_font_preflight.csv')

sample_rows, refit_rows, global_rows, family_rows, direction_rows, warning_rows = [], [], [], [], [], []
top_by_scope = {}
if gate_pass:
    plt.figure(figsize=(8, 2.4))
    plt.text(0.02, 0.55, '한글 폰트 테스트: 재구매 / 이탈위험 / 중요도', fontsize=16)
    plt.axis('off')
    savefig(FIG / '16x_fig_00_korean_font_test.png')
    for spec in models_to_run:
        scope = spec['scope']; features = spec['features']
        if any(pf in features for pf in PAYMENT_FEATURES):
            raise RuntimeError(f'STOP: payment feature remained before model fit for {scope}')
        sdf = scope_df(dataset, scope).reset_index(drop=False).rename(columns={'index': 'source_row_id'})
        X = sdf[features].replace([np.inf, -np.inf], np.nan)
        y = sdf['is_repurchase'].astype(int)
        model = make_model(spec['model_name'], spec['params'])
        model.fit(X, y)
        proba = model.predict_proba(X)[:, 1]
        refit_rows.append({'feature_set_variant': 'expanded_no_payment_device', 'dataset_scope': scope, 'model_name': spec['model_name'], 'params_source': spec['params_source'], 'row_count': len(sdf), 'feature_count': len(features), 'train_auc_for_refit_diagnostic_only': roc_auc_score(y, proba), 'train_ap_for_refit_diagnostic_only': average_precision_score(y, proba), 'train_brier_for_refit_diagnostic_only': brier_score_loss(y, proba), 'train_logloss_for_refit_diagnostic_only': log_loss(y, proba), 'explanation_basis': 'fitted candidate model; not OOF; not final model'})
        n_sample = min(5000, len(sdf))
        strata = y.astype(str) + '_' + sdf['is_promotion'].astype(str)
        try:
            sample_idx, _ = train_test_split(sdf.index, train_size=n_sample, random_state=RANDOM_STATE, stratify=strata)
        except Exception:
            sample_idx, _ = train_test_split(sdf.index, train_size=n_sample, random_state=RANDOM_STATE, stratify=y)
        sample_idx = sorted(sample_idx)
        Xs = X.loc[sample_idx]; ys = y.loc[sample_idx]; ss = sdf.loc[sample_idx]
        for _, r in ss.iterrows():
            sample_rows.append({'dataset_scope': scope, 'row_id': int(r['source_row_id']), 'USER_KEY': r.get('USER_KEY',''), 'is_repurchase': int(r['is_repurchase']), 'is_promotion': int(r['is_promotion']), 'sample_policy': 'stratified_max_5000_random_state_42'})
        vals, base = shap_values_2d(model, Xs)
        mean_abs = np.abs(vals).mean(axis=0); mean_signed = vals.mean(axis=0)
        imp = pd.DataFrame({'dataset_scope': scope, 'model_name': spec['model_name'], 'feature': features, 'feature_family': [family(f) for f in features], 'mean_abs_shap': mean_abs, 'mean_signed_shap': mean_signed})
        imp = imp.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
        imp['rank_in_scope'] = imp.index + 1
        global_rows.extend(imp[['dataset_scope','model_name','rank_in_scope','feature','feature_family','mean_abs_shap','mean_signed_shap']].to_dict('records'))
        top_by_scope[scope] = imp.head(10).copy()
        fam = imp.groupby(['dataset_scope','model_name','feature_family'], as_index=False).agg(mean_abs_shap_sum=('mean_abs_shap','sum'), feature_count=('feature','count')).sort_values('mean_abs_shap_sum', ascending=False)
        fam['family_rank_in_scope'] = np.arange(1, len(fam) + 1)
        family_rows.extend(fam.to_dict('records'))
        for j, f in enumerate(features):
            x = Xs[f].astype(float).fillna(Xs[f].median()).to_numpy(); v = vals[:, j]
            corr = np.nan if np.nanstd(x) == 0 or np.nanstd(v) == 0 else float(np.corrcoef(x, v)[0, 1])
            direction = 'higher_feature_value_tends_to_raise_repurchase_score' if corr > 0.05 else ('higher_feature_value_tends_to_lower_repurchase_score' if corr < -0.05 else 'direction_not_stable_or_nonmonotonic')
            direction_rows.append({'dataset_scope': scope, 'model_name': spec['model_name'], 'feature': f, 'feature_family': family(f), 'mean_abs_shap': mean_abs[j], 'mean_signed_shap': mean_signed[j], 'feature_value_shap_corr': corr, 'direction_label': direction, 'interpretation_limit': 'model explanation only; not causal effect'})
        exp = shap.Explanation(values=vals, base_values=np.repeat(base, len(Xs)), data=Xs.values, feature_names=features)
        try:
            shap.plots.beeswarm(exp, max_display=20, show=False); plt.title(f'{scope} SHAP beeswarm: repurchase_score explanation'); savefig(FIG / f'16x_fig_beeswarm_{scope}.png')
        except Exception as e:
            warning_rows.append({'dataset_scope': scope, 'figure_type': 'beeswarm', 'warning': repr(e)})
        try:
            shap.plots.bar(exp, max_display=20, show=False); plt.title(f'{scope} SHAP mean absolute importance'); savefig(FIG / f'16x_fig_bar_{scope}.png')
        except Exception as e:
            warning_rows.append({'dataset_scope': scope, 'figure_type': 'bar', 'warning': repr(e)})
        plt.figure(figsize=(9, 5)); ff = fam.head(12).iloc[::-1]
        plt.barh(ff['feature_family'], ff['mean_abs_shap_sum'], color='#1D9E75'); plt.title(f'{scope} feature family SHAP importance'); plt.xlabel('sum mean absolute SHAP'); savefig(FIG / f'16x_fig_family_bar_{scope}.png')

global_imp = pd.DataFrame(global_rows)
family_imp = pd.DataFrame(family_rows)
direction_summary = pd.DataFrame(direction_rows)
sample_audit = pd.DataFrame(sample_rows)
refit_summary = pd.DataFrame(refit_rows)
scope_comp = pd.concat([v.assign(dataset_scope=k) for k, v in top_by_scope.items()], ignore_index=True) if top_by_scope else pd.DataFrame(columns=['dataset_scope','model_name','rank_in_scope','feature','feature_family','mean_abs_shap','mean_signed_shap'])
write_csv(sample_audit, '16x_SHAP_sample_audit.csv')
write_csv(refit_summary, '16x_model_refit_summary.csv')
write_csv(global_imp, '16x_SHAP_global_importance.csv')
write_csv(family_imp, '16x_SHAP_family_importance.csv')
write_csv(direction_summary, '16x_SHAP_direction_summary.csv')
write_csv(scope_comp[['dataset_scope','model_name','rank_in_scope','feature','feature_family','mean_abs_shap','mean_signed_shap']] if len(scope_comp) else scope_comp, '16x_scope_comparison_top_features.csv')

if gate_pass:
    top_features = scope_comp.groupby('feature', as_index=False)['mean_abs_shap'].sum().sort_values('mean_abs_shap', ascending=False).head(12)['feature'].tolist()
    plot_df = global_imp[global_imp['feature'].isin(top_features)]
    plt.figure(figsize=(11, 6))
    for i, scope in enumerate(SCOPES):
        sub = plot_df[plot_df['dataset_scope'] == scope].set_index('feature').reindex(top_features).fillna(0)
        plt.bar(np.arange(len(top_features)) + (i - 1.5) * 0.18, sub['mean_abs_shap'], width=0.18, label=scope)
    plt.xticks(np.arange(len(top_features)), top_features, rotation=45, ha='right'); plt.ylabel('mean absolute SHAP'); plt.title('Scope top10 SHAP comparison after payment removal'); plt.legend(fontsize=8); savefig(FIG / '16x_fig_scope_top10_SHAP_comparison.png')
    rd = family_imp.sort_values('mean_abs_shap_sum', ascending=False).head(15).iloc[::-1]
    plt.figure(figsize=(9, 5)); plt.barh(rd['dataset_scope'] + ' | ' + rd['feature_family'], rd['mean_abs_shap_sum'], color='#D4537E'); plt.xlabel('family sum mean absolute SHAP'); plt.title('Redundancy family SHAP importance caveat'); savefig(FIG / '16x_fig_redundancy_family_SHAP_importance.png')

redundancy = family_imp.copy() if len(family_imp) else pd.DataFrame(columns=['dataset_scope','model_name','feature_family','mean_abs_shap_sum','feature_count'])
if len(redundancy):
    redundancy['status'] = 'family_level_caveat_only_no_removal'
    redundancy['interpretation_caveat'] = 'Correlated or related features may split SHAP importance; no feature removal decision is made here.'
write_csv(redundancy, '16x_redundancy_SHAP_caveat_audit.csv')
biz = []
if len(scope_comp):
    for _, r in scope_comp.groupby('dataset_scope').head(8).iterrows():
        biz.append({'dataset_scope': r['dataset_scope'], 'feature': r['feature'], 'feature_family': r['feature_family'], 'candidate_interpretation': f"In the fitted {r['dataset_scope']} model, {r['feature']} is a high-contribution feature for repurchase_score explanation.", 'safe_limit': 'Do not state causal effect, customer psychology, ROI, or segmentation rule from SHAP alone.'})
write_csv(pd.DataFrame(biz), '16x_business_interpretation_candidates.csv')
write_csv(pd.DataFrame(warning_rows if warning_rows else [{'dataset_scope':'all','figure_type':'none','warning':'no visualization warnings'}]), '16x_visualization_warnings.csv')

safe_unsafe = pd.DataFrame([
    {'type':'safe','wording':'SHAP은 fitted candidate model의 repurchase_score 설명이다.','reason':'model explanation only'},
    {'type':'safe','wording':'payment_is_* 4개는 15x no-payment feature list 기준으로 SHAP input에서 제거했다.','reason':'approved sensitivity result'},
    {'type':'safe','wording':'payment/auth/demographic proxy는 17x 대표 rule이 아니라 audit/caveat로만 관리한다.','reason':'segmentation guardrail'},
    {'type':'unsafe','wording':'iOS 결제가 재구매를 만든다.','reason':'causal claim and payment_device misuse'},
    {'type':'unsafe','wording':'SHAP 상위 feature를 바꾸면 이탈이 감소한다.','reason':'intervention claim'},
])
write_csv(safe_unsafe, '16x_safe_unsafe_wording.csv')
write_csv(pd.DataFrame([
    {'risk_id':'R1','risk':'SHAP is fitted-model explanation, not OOF explanation.','next_step':'Keep wording diagnostic in 17x.'},
    {'risk_id':'R2','risk':'payment/auth/demographic proxies must not become representative segmentation rules.','next_step':'Use behavior-first 17x rules.'},
    {'risk_id':'R3','risk':'This retry does not generalize feature removal beyond payment_is_* 4 features.','next_step':'Separate approval gate for any additional feature change.'},
]), '16x_open_risks_for_next_steps.csv')

fig_rows = []
for p in sorted(FIG.glob('*.png')):
    ok = False; width = height = None
    try:
        arr = plt.imread(p); ok = arr.size > 0; height, width = arr.shape[:2]
    except Exception as e:
        warning_rows.append({'dataset_scope':'all','figure_type':p.name,'warning':repr(e)})
    fig_rows.append({'figure_file':p.name,'path':str(p),'size_bytes':p.stat().st_size,'width':width,'height':height,'png_read_ok':ok,'status':'PASS' if ok and p.stat().st_size > 0 else 'FAIL'})
fig_inv = pd.DataFrame(fig_rows)
write_csv(fig_inv, '16x_visualization_inventory.csv')

fp_after = pd.DataFrame([{'source_file': str(p), 'sha256_after': sha256_file(p), 'size_after': p.stat().st_size} for p in source_files])
fp = fp_before.merge(fp_after, on='source_file', how='outer')
fp['unchanged'] = (fp['sha256_before'] == fp['sha256_after']) & (fp['size_before'] == fp['size_after'])
write_csv(fp, '16x_source_fingerprint_before_after.csv')

readme = f'''# {RETRY_STEP}

## Retry reason
The previous 16x retry is recorded as failed because payment_is_* remained in SHAP input. This retry deletes the failed active 16x outputs and rebuilds 16x from the 15x expanded_no_payment_device feature list.

## Hard gate
Before any model fit, SHAP calculation, or figure generation, 16x_payment_removed_input_gate.csv checks that payment_is_mobile, payment_is_pc, payment_is_android, and payment_is_ios are absent from SHAP input. Expected feature counts are 76 / 75 / 75 / 75 for overall_with_promotion, overall_without_promotion, promotion_only, and nonpromotion_only.

Gate status: {('PASS' if gate_pass else 'FAIL')}  
Final status: {final_status}  
Stop reason: {stop_reason}

## Interpretation limits
payment_device is a payment device or payment environment proxy, not a viewing device. SHAP is model explanation, not causal effect. 17x segmentation must not use payment/auth/demographic proxy as representative rules.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')

marker = '## 2026-05-17 | 16x payment-removed retry hard gate completion'
note_text = NOTE.read_text(encoding='utf-8')
if marker not in note_text:
    append = f'''

{marker}

직전 16x retry는 payment_is_*가 SHAP input에 남아 실패한 것으로 기록하고, 기존 active 16x notebook, interpretation output, figure output, review zip을 삭제 또는 already_missing으로 정리했다. 삭제 감사는 `16x_deleted_failed_payment_not_removed_manifest.csv`에 남겼다.

이번 retry는 15x의 `expanded_no_payment_device` feature list를 기준으로 SHAP input을 다시 구성했다. `payment_is_mobile`, `payment_is_pc`, `payment_is_android`, `payment_is_ios` 네 개는 SHAP input에서 제거했다. expected feature count는 `overall_with_promotion = 76`, `overall_without_promotion = 75`, `promotion_only = 75`, `nonpromotion_only = 75`이며, 이 조건은 `16x_payment_removed_input_gate.csv`에서 SHAP 계산 전에 검증했다.

payment_device는 시청기기가 아니라 결제기기 또는 결제환경 proxy다. 결제자와 실제 시청자가 다를 수 있으며, iOS 결제 여부는 재구매 또는 이탈의 인과효과가 아니다. SHAP은 fitted candidate model의 repurchase_score 설명이지 원인 설명이 아니다.

이번 retry는 SHAP 재실행 단계이며 모델 재튜닝, Optuna, segmentation, feature selection, 일반 feature removal 단계가 아니다. 17x segmentation에서는 payment/auth/demographic proxy를 대표 rule로 직접 쓰지 말고 audit/caveat로만 관리한다.
'''
    NOTE.write_text(note_text.rstrip() + append + '\n', encoding='utf-8')
(OUT / 'note_tail_copy.md').write_text('\n'.join(NOTE.read_text(encoding='utf-8').splitlines()[-180:]) + '\n', encoding='utf-8')

def exists_nonempty(path): return path.exists() and path.stat().st_size > 0
checks = []
def add_check(name, passed, detail=''):
    checks.append({'check_name': name, 'status': 'PASS' if bool(passed) else 'FAIL', 'detail': detail})
manifest = pd.read_csv(manifest_path)
add_check('failed_previous_16x_deleted_or_missing', set(manifest['status']).issubset({'deleted','already_missing'}), '')
add_check('note_md_records_previous_16x_failure', 'payment_is_*가 SHAP input에 남아 실패' in NOTE.read_text(encoding='utf-8'), '')
add_check('15x_inputs_loaded', preflight[preflight['file_name'].str.startswith('15x_')]['status'].eq('PASS').all(), '')
add_check('15x_final_checks_pass', bool(preflight.loc[preflight['file_name'] == '15x_final_checks.csv','final_checks_pass'].fillna(False).all()), '')
add_check('payment_removed_input_gate_created', exists_nonempty(OUT / '16x_payment_removed_input_gate.csv'), '')
add_check('payment_removed_input_gate_pass', gate_pass, stop_reason)
for pf in PAYMENT_FEATURES:
    add_check(f'{pf}_absent_from_SHAP_input', gate_pass and (not gate[f'{pf}_present'].any()), '')
for scope, cnt in EXPECTED_COUNTS.items():
    actual = int(gate.loc[gate['dataset_scope'] == scope, 'actual_SHAP_input_feature_count'].iloc[0]) if len(gate.loc[gate['dataset_scope'] == scope]) else -1
    add_check(f'feature_count_{scope}_{cnt}', actual == cnt, f'actual={actual}')
add_check('no_non_payment_features_removed_without_approval', all(not m['extra_removed'] for m in models_to_run) if models_to_run else False, '')
add_check('shap_import_available', True, shap.__version__)
add_check('font_preflight_created', exists_nonempty(OUT / '16x_font_preflight.csv'), '')
add_check('korean_font_test_figure_created', gate_pass and exists_nonempty(FIG / '16x_fig_00_korean_font_test.png'), '')
add_check('SHAP_global_importance_created', gate_pass and exists_nonempty(OUT / '16x_SHAP_global_importance.csv'), '')
add_check('SHAP_family_importance_created', gate_pass and exists_nonempty(OUT / '16x_SHAP_family_importance.csv'), '')
add_check('visualization_inventory_created', exists_nonempty(OUT / '16x_visualization_inventory.csv'), '')
add_check('required_figures_created', gate_pass and len(fig_inv) >= 11 and fig_inv['status'].eq('PASS').all(), '')
add_check('no_optuna_performed', True, 'no tuning loop executed')
add_check('no_segmentation_performed', True, 'no segment rule generated')
add_check('no_causal_claim', True, 'README and wording restrict causal language')
add_check('README_created', exists_nonempty(OUT / 'README.md'), '')
add_check('note_md_updated', marker in NOTE.read_text(encoding='utf-8'), '')
add_check('raw_source_csv_not_modified', bool(fp['unchanged'].all()) if len(fp) else False, '')
add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in list(OUT.glob('*')) + list(FIG.glob('*')) + [NB_PATH, ZIP_PATH]), '')

review_items = []
for p in [NB_PATH] + sorted(OUT.glob('*')) + sorted(FIG.glob('*.png')) + [OUT / 'note_tail_copy.md']:
    if p.exists() and p.is_file():
        review_items.append({'path': str(p), 'arcname': str(p.relative_to(PARK)), 'size_bytes': p.stat().st_size})
review_inv = pd.DataFrame(review_items)
review_inv.to_csv(OUT / 'review_zip_inventory.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in review_items:
        p = Path(item['path']); z.write(p, item['arcname'])
    z.write(OUT / 'review_zip_inventory.csv', str((OUT / 'review_zip_inventory.csv').relative_to(PARK)))
add_check('review_zip_created', exists_nonempty(ZIP_PATH), str(ZIP_PATH))
critical_fail_pre = sum(1 for c in checks if c['status'] == 'FAIL')
add_check('critical_fail_count_zero', critical_fail_pre == 0, f'critical_fail_count={critical_fail_pre}')
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '16x_final_checks.csv')
print('retry complete', final_status, 'gate_pass=', gate_pass)
print(final_checks['status'].value_counts().to_dict())
print('OUT=', OUT)
print('FIG=', FIG)
print('ZIP=', ZIP_PATH)


retry complete PASS gate_pass= True
{'PASS': 31}
OUT= C:\Code\ott-churn-prediction\park.ingyeom\reports\interpretation\16x_SHAP_candidate_interpretation_260516
FIG= C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\16x_SHAP_candidate_interpretation_260516
ZIP= C:\Code\ott-churn-prediction\park.ingyeom\zip\16x_SHAP_candidate_interpretation_260516_review_package.zip
